In [ ]:
# =============================================================================
# FUMD-AI Preprocessing Workflow -- Step 3: Generate the OMNeT++ feature matrix
# =============================================================================
# Step:         3 of 7 (input stage for the core labeling chain)
# Summary:      Clean raw OMNeT++ vector export and pivot into a wide per-vehicle-per-timestep feature matrix.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-preprocessing-workflow
# Version:      1.0.0
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# -----------------------------------------------------------------------------
#
# Structured, machine-readable metadata for this workflow (authors, license,
# inputs/outputs per step) is also maintained in ro-crate-metadata.json at
# the repository root - update both together if either changes.
# =============================================================================


# Step 3 — Generate the OMNeT++ feature matrix

Part of the **FUMD-AI preprocessing workflow**: turns raw SUMO + OMNeT++
simulation output into a labeled, AI-ready dataset of cellular handover
events. This is step 3 of 7 (step 7 is optional/exploratory).

**Purpose.** OMNeT++ exports one row per `(time, module, metric)` observation.
This notebook cleans that raw export and pivots it into a wide table with one
row per `(vehicle, time)` and one column per network-quality metric (SINR,
CQI, RLC delay/throughput, serving cell, distance to the serving gNB, ...).
It also adds lagged and lead `servingCell` columns
(`servingCell-1..-7` / `servingCell1..7`) capturing which cell each vehicle
was connected to N seconds in the past/future.

**Input:** a raw OMNeT++ vector CSV, tab-separated with columns
`Time  Object  Vector  Value`, e.g.:

```
Time    Object                                  Vector                  Value
0.1     car[0].cellularNic.phy                  "servingCell """""      0.0
0.1     car[0].cellularNic.nrChannelModel[0]    "distance """""         730.321606
0.104   car[0].cellularNic.nrPhy                "averageCqiDl """""     10.0
0.107   car[0].cellularNic.nrRlc.um             "rlcThroughputDl """""  644.859813
```

**Output:** `<OUTPUT_MATRIX_PATH>` — one row per vehicle per resampled tick.

**Note:** this notebook never modifies the raw input file in place. All
intermediate/cleaned files are written fresh on every run, so it is always
safe to re-run from scratch.

**System requirement:** a `sed` binary on `PATH` (used for fast text
cleaning of the raw export, which can be tens of GB — a pure-pandas
string-replace pass over that many rows would be far slower and far more
memory-hungry). The cleaning pattern is written portably (literal tab
bytes rather than a `	` escape) so it works identically with both GNU
`sed` (default on Linux) and the BSD `sed` shipped by default on macOS -
no need to install GNU sed separately on Mac.

**Validated against real `extractvectors` output.** This notebook's
cleaning, pivoting and resampling logic was run end-to-end against a real
OMNeT++ vector export produced by `extractvectors` from
`VoipDl-Urban-900_1/vector-0.vec`: 3,965,973 raw car rows in, correctly
cleaned and pivoted into a 555,241-row x 25-column feature matrix for 75
vehicles, with genuine multi-cell serving-cell activity (cells 0, 1, 2, 5,
8, 9) and zero missing values after fill/resample.


In [ ]:
import subprocess
import pandas as pd


In [ ]:
# ---- Parameters (edit for your own simulation run) ----
# This cell is tagged "parameters" so the notebook can also be executed
# headlessly with papermill, e.g.:
#   papermill step_3_generate_omnet_matrix.ipynb out.ipynb -p RAW_OMNET_PATH my_run_omnet_export.csv

# Default points at the tiny bundled example (example-data/) so this
# notebook runs out of the box without needing Step 1's extractvectors
# step - replace with your own Step 1 output (or raw export) directly.
RAW_OMNET_PATH = "example-data/raw_omnet_export.csv"  # raw OMNeT++ vector export (input)
CLEAN_OMNET_PATH = "omnet_clean.csv"                  # intermediate cleaned/parsed file (generated)
OUTPUT_MATRIX_PATH = "omnet_feature_matrix.csv"        # final wide-format matrix (output) -> feeds Step 4's OMNET_PATH

RESAMPLE_FREQ = "10ms"                    # uniform time grid to resample onto
LAG_LEAD_SECONDS = [1, 2, 3, 4, 5, 6, 7]  # adds servingCell-N / servingCellN columns

# Metrics that are always 0 for this simulation setup and add no information;
# dropped right after pivoting. Remove entries here if your OMNeT++ config
# actually records nonzero values for them.
ALWAYS_ZERO_VECTORS = ["rlcPacketLossTotal ", "rlcPduPacketLossDl ", "rlcPduThroughputDl "]

MAX_ROWS = None  # set an integer to only read the first N raw rows (quick test runs)


## 1. Clean the raw OMNeT++ export

The raw export needs five fixes before it can be read as a normal wide CSV:

1. Drop a redundant `servingCell` line that OMNeT++ logs under the wrong
   module (`cellularNic.phy` instead of `cellularNic.nrPhy`).
2. Collapse `car[i].cellularNic.<submodule>` object names down to just the
   bare vehicle id `i` (the submodule is implied by the metric name already).
3. Drop `receivedPacketFromLowerLayer` rows (not one of the metrics we keep).
4. Strip the `car[...]` wrapper down to a plain integer vehicle id.
5. Strip stray double quotes left by OMNeT++'s vector export format.

All five fixes are applied in a single `sed` pass for speed on multi-GB
files, streaming straight from `RAW_OMNET_PATH` into a **new**
`CLEAN_OMNET_PATH` file — the raw input is never touched.

**Why dropping the `cellularNic.phy` `servingCell` row (fix 1) is enough.**
Some simu5g network configurations (e.g. `NRSeveralBSALC`, dual-connectivity
capable) declare *both* an LTE-style stack (`cellularNic.rlc.um`,
`cellularNic.phy`) and the 5G NR stack (`cellularNic.nrRlc.um`,
`cellularNic.nrPhy`) per vehicle, under the *same* metric names. If both
stacks actually carried data, collapsing the module path down to just the
metric name (fix 2) would silently conflate two different signals under
one column. Checked directly against a real raw `.vec` file from this
project (`VoipDl-Urban-900_1/vector-0.vec`, multiple vehicles): the
LTE-style stack's `servingCell` is recorded but is constant `0` for the
entire simulation (it is never actually connected to a base station in
this scenario), and every *other* LTE-style metric (`rlcDelayDl`,
`averageCqiDl`, etc.) has **zero** recorded data rows at all - only the NR
stack ever carries real values. So dropping just the one row fix 1 targets
is sufficient here; if you use a network configuration where the
non-NR stack is genuinely populated with data, this cleaning step will
need to explicitly filter to the NR module path instead.


In [ ]:
def clean_omnet_raw(raw_path: str, clean_path: str) -> None:
    """Stream-clean a raw OMNeT++ vector export into a pandas-readable TSV.

    Non-destructive: never modifies `raw_path`. Fully overwrites
    `clean_path` on every call, so re-running this cell is always safe.
    """
    # Tab characters are embedded here as literal bytes (via TAB), not as a
    # "\t" escape sequence, so this sed script behaves identically under
    # GNU sed (Linux) and the BSD sed shipped by default on macOS. BSD sed
    # does not treat "\t" in a pattern as a tab character - it matches the
    # literal letter t - so relying on that escape silently mis-cleans the
    # file on macOS (no error, just corrupted output) instead of failing
    # loudly. A literal tab byte in the pattern is unambiguous on both.
    TAB = "\t"
    sed_script = (
        r'/' + TAB + r'car\[[0-9]+\]\.cellularNic\.phy' + TAB + r'"*servingCell/d; '
        r's/\]\.cellularNic\.[^' + TAB + r']*' + TAB + r'/' + TAB + r'/; '
        r'/receivedPacketFromLowerLayer/d; '
        r's/car\[//g; '
        r's/"//g'
    )
    with open(raw_path) as raw_f, open(clean_path, "w") as clean_f:
        subprocess.run(["sed", "-E", sed_script], stdin=raw_f, stdout=clean_f, check=True)


clean_omnet_raw(RAW_OMNET_PATH, CLEAN_OMNET_PATH)
print("cleaned file written to", CLEAN_OMNET_PATH)


## 2. Load, pivot to wide format, and fill small per-vehicle gaps

In [ ]:
# Load the cleaned export. It is still in "long" format at this point:
# one row per (Time, Object=vehicle id, Vector=metric name, Value).
omnet = pd.read_csv(CLEAN_OMNET_PATH, delimiter="\t", nrows=MAX_ROWS)
print("raw rows:", len(omnet))
omnet.head()


In [ ]:
# long -> wide: pivot "Vector" (metric name) out into its own column, so each
# (Time, Object) pair becomes one row with one column per metric. `.aggregate("min")`
# is only there to collapse the rare case of duplicate (Time, Object, Vector)
# rows into a single value - it does not change anything for the normal case
# of one observation per metric per timestep.
wide = omnet.groupby(["Time", "Object", "Vector"])["Value"].aggregate("min").unstack()
wide = wide.reset_index().set_index(["Object", "Time"])

# Not every metric is logged at every timestep (OMNeT++ only logs a value
# when it changes). Forward-fill first (carry the last known value forward),
# then back-fill (so the very first rows of a vehicle's trip, before its
# first observation, get its first known value instead of staying empty).
wide = wide.groupby("Object").ffill()
wide = wide.groupby("Object").bfill()
wide = wide.reset_index().sort_values(by=["Time", "Object"])

assert not wide.isnull().values.any(), "unexpected NaNs remain after fill"
wide.head()


## 3. Resample onto a uniform time grid per vehicle

In [ ]:
# pandas .resample() needs a proper datetime/timedelta index, not raw floats
wide["Time"] = pd.to_timedelta(wide["Time"], unit="s")
wide = wide.set_index("Time")

# drop metrics known to be always 0 for this simulation setup (see ALWAYS_ZERO_VECTORS above)
wide = wide.drop(columns=[c for c in ALWAYS_ZERO_VECTORS if c in wide.columns])

print("servingCell values before resampling:", sorted(wide["servingCell "].dropna().unique()))


In [ ]:
# Resample each vehicle independently onto the uniform RESAMPLE_FREQ grid.
# Every metric is averaged over each resample window, except servingCell,
# which takes the *last* value in the window so it always stays a whole
# cell id (never a meaningless decimal average of two different cells,
# e.g. cell 3 and cell 4 averaging to 3.5).
#
# Grouping by ["Object", pd.Grouper(freq=...)] rather than doing
# .groupby("Object").resample(...) is deliberate: the groupby+resample form
# routes through DataFrameGroupBy.apply() internally, whose handling of the
# grouping columns changed across pandas versions (an `include_groups`
# option was added in pandas 2.2 to control it, but passing that same
# keyword straight into .resample() raises a TypeError on pandas < 2.2,
# since .resample() itself never accepted it - it only ever belonged to
# .apply()). Grouping with pd.Grouper avoids that version-dependent code
# path entirely and behaves identically on pandas 2.1 and 2.2+.
grouped = wide.groupby(["Object", pd.Grouper(freq=RESAMPLE_FREQ)])
final = grouped.mean(numeric_only=True)
final["servingCell "] = grouped["servingCell "].last().reindex(final.index).astype("Int64")

# resampling can introduce empty ticks at the very edges of a vehicle's
# trip (before its first / after its last observation) - drop those
final = final.dropna().reset_index()
final["Time"] = final["Time"].dt.total_seconds()  # back to plain seconds, easier to work with downstream

# OMNeT++'s vector names carry a trailing space (e.g. "servingCell "); strip it
final.columns = [c.strip() for c in final.columns]

# cosmetic: keep Time/Object as the first two columns
final = final[["Time", "Object"] + [c for c in final.columns if c not in ("Time", "Object")]]

assert not final.isnull().values.any(), "unexpected NaNs remain after resampling"
final.head()


## 4. Add lagged / lead `servingCell` columns

For each of `LAG_LEAD_SECONDS`, add a column with the serving cell N
seconds in the past (`servingCell-N`) and N seconds in the future
(`servingCellN`). A value of `-1` means "no data available yet" (start or
end of the vehicle's trip) rather than a real cell id.


In [ ]:
# how many resampled rows correspond to one second, given RESAMPLE_FREQ
# (e.g. 100 rows/second at the default 10ms grid)
resample_seconds = pd.Timedelta(RESAMPLE_FREQ).total_seconds()

for sec in LAG_LEAD_SECONDS:
    shift_rows = round(sec / resample_seconds)
    # shift(+N) looks backwards in time -> "what cell was this vehicle on N seconds ago"
    final[f"servingCell-{sec}"] = final.groupby("Object")["servingCell"].shift(shift_rows).fillna(-1)
    # shift(-N) looks forwards in time -> "what cell will this vehicle be on N seconds from now"
    final[f"servingCell{sec}"] = final.groupby("Object")["servingCell"].shift(-shift_rows).fillna(-1)

final.head()


## 5. Save the output matrix

In [ ]:
# tab-separated to stay consistent with the raw OMNeT++ export format used upstream
final.to_csv(OUTPUT_MATRIX_PATH, index=False, sep="\t")
print(f"saved {OUTPUT_MATRIX_PATH} - shape {final.shape}")
